# Implementing an RNN for Text Generation

## Task: Recurrent Neural Networks (RNNs) can generate sequences of text. You will train an LSTM-based RNN to predict the next character in a given text dataset.

1.	Load a text dataset (e.g., "Shakespeare Sonnets", "The Little Prince").
2.	Convert text into a sequence of characters (one-hot encoding or embeddings).
3.	Define an RNN model using LSTM layers to predict the next character.
4.	Train the model and generate new text by sampling characters one at a time.
5.	Explain the role of temperature scaling in text generation and its effect on randomness.


In [47]:
import tensorflow as tf
import numpy as np

In [48]:
path = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)

text = open(path, "rb").read().decode("utf-8")

print("Number of characters:", len(text))
print(text[:1000])

Number of characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread

In [49]:
# Creating charater vocabulary
vocab = sorted(set(text))

vocab_size = len(vocab)

print("Number of unique characters:", vocab_size)
print(vocab)


Number of unique characters: 65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [50]:
# Creating mappings
char_to_idx = {
    char: i
    for i, char in enumerate(vocab)
}

idx_to_char = np.array(vocab)
print("Character to index mapping:", char_to_idx)
print("Index to character mapping:", idx_to_char)

Character to index mapping: {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
Index to character mapping: ['\n' ' ' '!' '$' '&' "'" ',' '-' '.' '3' ':' ';' '?' 'A' 'B' 'C' 'D' 'E'
 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S' 'T' 'U' 'V' 'W'
 'X' 'Y' 'Z' 'a' 'b' 'c' 'd' 'e' 'f' 'g' 'h' 'i' 'j' 'k' 'l' 'm' 'n' 'o'
 'p' 'q' 'r' 's' 't' 'u' 'v' 'w' 'x' 'y' 'z']


In [51]:
# Convert book into integers
text_as_int = np.array([
    char_to_idx[c]
    for c in text
])

print(text[:50])
print(text_as_int[:50])

First Citizen:
Before we proceed any further, hear
[18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56 43  1 61 43
  1 54 56 53 41 43 43 42  1 39 52 63  1 44 59 56 58 46 43 56  6  1 46 43
 39 56]


In [52]:
# Define the sequence length and create training examples
seq_length = 100
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

sequences = char_dataset.batch(
    seq_length + 1,
    drop_remainder=True
)

def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]

    return input_text, target_text

dataset = sequences.map(split_input_target)

In [53]:
for input_example, target_example in dataset.take(1):

    print("Input:")
    print("".join(idx_to_char[input_example.numpy()]))

    print("\nTarget:")
    print("".join(idx_to_char[target_example.numpy()]))

Input:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

Target:
irst Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You 


In [54]:
# One hot encode the input and target sequences
def one_hot_encode(input_text, target_text):

    input_one_hot = tf.one_hot(
        input_text,
        depth=vocab_size
    )

    target_one_hot = tf.one_hot(
        target_text,
        depth=vocab_size
    )

    return input_one_hot, target_one_hot

dataset = dataset.map(one_hot_encode)


In [55]:
# View the one-hot encoded input and target sequences
for input_example, target_example in dataset.take(1):
    print("Input (one-hot encoded):")
    print(input_example.numpy())

    print("\nTarget (one-hot encoded):")
    print(target_example.numpy())
    

Input (one-hot encoded):
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

Target (one-hot encoded):
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]]


In [56]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

for input_batch, target_batch in dataset.take(1):
    print("Input shape:", input_batch.shape)
    print("Target shape:", target_batch.shape)

Input shape: (64, 100, 65)
Target shape: (64, 100, 65)


In [57]:
# Define the LSTM model
model = tf.keras.Sequential([
    
    tf.keras.layers.Input(
        shape=(None, vocab_size)
    ),

    tf.keras.layers.LSTM(
        256,
        return_sequences=True
    ),

    tf.keras.layers.Dense(
        vocab_size
    )
])

In [58]:
# View the model summary
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, None, 256)      │       329,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 65)       │        16,705 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 346,433 (1.32 MB)

 Trainable params: 346,433 (1.32 MB)

 Non-trainable params: 0 (0.00 B)

In [59]:
loss_function = tf.keras.losses.CategoricalCrossentropy(
    from_logits=True
)

model.compile(
    optimizer="adam",
    loss=loss_function,
    metrics=["accuracy"]
)

In [60]:
# Train the model
EPOCHS = 50

history = model.fit(
    dataset,
    epochs=EPOCHS
)

Epoch 1/50


/Users/aryamanshrestha/Neural Network and Deep Learning/coding/Home-Assignment-Neural-Network-and-Deep-Learning/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 58ms/step - accuracy: 0.1949 - loss: 3.1197
Epoch 2/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 58ms/step - accuracy: 0.3314 - loss: 2.4069
Epoch 3/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 58ms/step - accuracy: 0.3713 - loss: 2.2124
Epoch 4/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 57ms/step - accuracy: 0.3959 - loss: 2.1068
Epoch 5/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 57ms/step - accuracy: 0.4163 - loss: 2.0263
Epoch 6/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 57ms/step - accuracy: 0.4314 - loss: 1.9607
Epoch 7/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 58ms/step - accuracy: 0.4438 - loss: 1.9065
Epoch 8/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 58ms/step - accuracy: 0.4548 - loss: 1.8605
Epoch 9/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - accuracy: 0.4651 - loss: 1.8225
Epoch 10/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 58ms/step - accuracy: 0.4739 - loss: 1.7897
Epoch 11/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 58ms/step - accuracy: 0.4815 - loss: 1.7606
Epoch 12/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 

In [61]:
#Model Evaluation
accuracy = history.history['accuracy'][-1]
print(f"Final training accuracy: {accuracy*100:.4f}")

Final training accuracy: 59.3689


In [62]:
#Generating the text using the trained model

def generate_text(model,start_string,num_generate=500,temperature=1.0):
    generated_text = start_string

    input_chars = [
        char_to_idx[char]
        for char in start_string
        if char in char_to_idx
    ]

    for _ in range(num_generate):

        # Keep only the last 100 characters
        input_chars = input_chars[-seq_length:]

        # Convert to tensor
        input_tensor = tf.convert_to_tensor(
            [input_chars]
        )

        # One-hot encoding
        input_one_hot = tf.one_hot(
            input_tensor,
            depth=vocab_size
        )

        # Get predictions
        predictions = model(
            input_one_hot,
            training=False
        )

        # Get prediction for the last character
        predictions = predictions[:, -1, :]

        # Apply temperature
        predictions = predictions / temperature

        # Sample a character
        predicted_id = tf.random.categorical(
            predictions,
            num_samples=1
        )[0, 0].numpy()

        # Convert number back to character
        predicted_char = idx_to_char[predicted_id]

        # Add predicted character
        generated_text += predicted_char

        # Add character to next input
        input_chars.append(predicted_id)

    return generated_text

In [65]:
generated = generate_text(
    model,
    start_string="First Citizen:",
    num_generate=500,
    temperature=0.5
)

print(generated)

First Citizen:
Of what we have here, they long consent to do.
O her not you there was the first for my comple
Than the news in some with her speak in possess.

KING RICHARD II:
Why then the lands are for the fault of my life.

BENVOLIO:
Who stays if you well another death?

AUTOLYCUS:
I know not a troops of my lord.

ANGELO:
They do you good to him: better he will part her.

BRUTUS:
Lay her and this is the name of the destry.

HORTENSIO:
Madam, the great done and dispose bedome me,
That every speak not my lor


##### Explain the role of temperature scaling in text generation and its effect on randomness.

##### Answer:

Temperature scaling controls the randomness of the text generated by the model. A low temperature makes the model choose the most likely characters, which produces more predictable and repetitive text. A medium temperature provides a good balance between accuracy and variety. A high temperature increases randomness by giving less likely characters a greater chance of being selected, which can make the text more creative but may also produce meaningless or incorrect output.

For example, if the model predicts the next character after “The ki”, a low temperature may strongly choose “n” to continue toward “The king”, while a higher temperature might sometimes choose a less likely character and produce a more unusual continuation. Therefore, lower temperature means less randomness, while higher temperature means more randomness.